# 2 — View operations

Every operation returns a **new tensor sharing the same buffer** — only the
layout changes. Each section shows the internals before and after, with
physical coordinates throughout.

Working example: a 4×6 sensor tile, 6.5 um pixel pitch on both axes
(int64 values, so `i` has byte stride 48 and `j` stride 8).

In [1]:
import numpy as np
from nbhelp import show  # also puts tensorlib on sys.path
from tensorlib import Dim, Layout, Tensor, chart, q

In [2]:
arr = np.arange(24, dtype=np.int64).reshape(4, 6)
t = Tensor.from_numpy(arr, ("i", "j")).with_charts(i=("0 um", "6.5 um"), j=("0 um", "6.5 um"))
show(t, "the base tensor")

-- the base tensor
Tensor[int64] on Buffer(192B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i           48      0      4      4  pos[i: 0 um step 6.5 um]
  j            8      0      6      6  pos[j: 0 um step 6.5 um]
  numel=24  footprint=(0, 192)  injectivity=injective


## slice — shrinks domains, touches nothing else

Bounds can be physical. Because coordinates are raw (D3), a slice changes
**only** `[start, stop)`: strides, offset, and the chart are untouched —
pixel `j=19.5 um` is still called `j=19.5 um` afterwards.

In [3]:
s = t.slice(j=(q("13 um"), q("32.5 um")))
show(s, "t.slice(j=(13 um, 32.5 um))  — lattice [2, 5)")
print("same buffer:", s.buffer is t.buffer)
print(s.item(i=0, j=q("19.5 um")), "==", arr[0, 3])

-- t.slice(j=(13 um, 32.5 um))  — lattice [2, 5)
Tensor[int64] on Buffer(192B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i           48      0      4      4  pos[i: 0 um step 6.5 um]
  j            8      2      5      3  pos[j: 0 um step 6.5 um]
  numel=12  footprint=(16, 184)  injectivity=injective
same buffer: True
3 == 3


## select — drops a dim into the offset

Fixing `i = 13 um` (lattice 2) folds `stride_i · 2 = 96` into the offset and
removes the dim.

In [4]:
sel = t.select(i=q("13 um"))
show(sel, "t.select(i=13 um)")
sel.to_numpy()

-- t.select(i=13 um)
Tensor[int64] on Buffer(192B @ cpu)
  offset : 96 bytes
  dim     stride  start   stop   size  chart
  j            8      0      6      6  pos[j: 0 um step 6.5 um]
  numel=6  footprint=(96, 144)  injectivity=injective


array([12, 13, 14, 15, 16, 17])

## shift vs recenter — two different relabelings

`shift` relabels the **lattice**; the chart's origin compensates, so physical
labels stay glued to the data. It is a storage-side rename — the physics does
not move.

In [5]:
sh = t.shift(i=100)
show(sh, "t.shift(i=100): lattice moved, physics glued")
print(sh.item(i=102, j=0), "==", arr[2, 0])
print("physical indexing unchanged:", sh.item(i=q("13 um"), j=0) == t.item(i=q("13 um"), j=0))

-- t.shift(i=100): lattice moved, physics glued
Tensor[int64] on Buffer(192B @ cpu)
  offset : -4800 bytes
  dim     stride  start   stop   size  chart
  i           48    100    104      4  pos[i: -650 um step 6.5 um]
  j            8      0      6      6  pos[j: 0 um step 6.5 um]
  numel=24  footprint=(0, 192)  injectivity=injective
12 == 12
physical indexing unchanged: True


`recenter` is the converse: it moves the **physical frame** (origin += Δ) and
touches nothing else — not the lattice, not the data.

In [6]:
rc = t.recenter(i=q("100 um"))
show(rc, "t.recenter(i=100 um): frame moved, lattice untouched")
print(rc.phys("i", 0), "   data unchanged:", rc.item(i=0, j=0) == arr[0, 0])

-- t.recenter(i=100 um): frame moved, lattice untouched
Tensor[int64] on Buffer(192B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i           48      0      4      4  pos[i: 100 um step 6.5 um]
  j            8      0      6      6  pos[j: 0 um step 6.5 um]
  numel=24  footprint=(0, 192)  injectivity=injective
100 um    data unchanged: True


## rename — identity is the name

The chart rides along with the dim.

In [7]:
show(t.rename(i="row", j="col"), 'rename(i="row", j="col")')

-- rename(i="row", j="col")
Tensor[int64] on Buffer(192B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  row         48      0      4      4  pos[i: 0 um step 6.5 um]
  col          8      0      6      6  pos[j: 0 um step 6.5 um]
  numel=24  footprint=(0, 192)  injectivity=injective


## there is no permute

Dim order carries no addressing meaning (D5), so there is nothing to permute.
Export order is a property of the boundary (`to_numpy(order=...)`); order-free
identity is a query (`canonical()`) — and it sees through unit spelling,
because Quantity equality is semantic.

In [8]:
t.to_numpy(order=("j", "i")).shape  # export order chosen at the boundary

(6, 4)

In [9]:
a = Layout(
    (
        Dim("i", 48, 0, 4, chart("0 um", "6.5 um")),
        Dim("j", 8, 0, 6, chart("0 um", "6.5 um")),
    )
)
b = Layout(
    (
        Dim("j", 8, 0, 6, chart("0 nm", "6500 nm")),
        Dim("i", 48, 0, 4, chart("0 um", "6.5 um")),
    )
)
print("dataclass equality sees order & spelling:", a == b)
print("canonical() erases both:                 ", a.canonical() == b.canonical())

dataclass equality sees order & spelling: False
canonical() erases both:                  True


## flip — storage reversal; the physics is invariant

The stride negates and the offset re-anchors; the chart follows the data
(origin re-anchors, step goes negative), so every datum keeps its physical
label.

In [10]:
f = t.flip("j")
show(f, 't.flip("j")')
print("label glued to datum:", f.item(i=0, j=q("13 um")) == t.item(i=0, j=q("13 um")))
f.to_numpy()[0]

-- t.flip("j")
Tensor[int64] on Buffer(192B @ cpu)
  offset : 40 bytes
  dim     stride  start   stop   size  chart
  i           48      0      4      4  pos[i: 0 um step 6.5 um]
  j           -8      0      6      6  pos[j: 32.5 um step -6.5 um]
  numel=24  footprint=(0, 192)  injectivity=injective
label glued to datum: True


array([5, 4, 3, 2, 1, 0])

## repeat — a stride-0 dimension

Every coordinate along `copies` aliases the same storage, and the analysis
*proves* it. The new dim is uncharted by default (attach one with
`with_charts` if the copies have physical meaning).

In [11]:
r = t.select(i=q("0 um")).repeat("copies", 3)
show(r, 'select(i=0 um).repeat("copies", 3)')

-- select(i=0 um).repeat("copies", 3)
Tensor[int64] on Buffer(192B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  j            8      0      6      6  pos[j: 0 um step 6.5 um]
  copies       0      0      3      3  
  numel=18  footprint=(0, 48)  injectivity=aliased
